In [1]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

from open_dataset_store import quick_start

# Initialize the OpenDatasetStore
store = quick_start('.', backend='local')

# Load data from the provided raw data file
data_path = "raw_data/experiments/entry_0009_zone_002_1785989318_data.csv"
df = pd.read_csv(data_path)
df['timestamp'] = pd.to_datetime(df['timestamp'])


Store initialised at: . (Backend: local)


In [2]:
# ==========================================
# Data Cleaning & Outlier Removal
# ==========================================

# Select columns to clean
cols_to_clean = [
    'outside_t', 'room_1_t', 'room_2_t', 'room_3_t', 'supply_t', 
    'outside_h', 'room_1_h', 'room_2_h', 'room_3_h', 'supply_h',
    'outside_c', 'room_1_c', 'room_2_c', 'room_3_c', 'supply_c'
]

# Use OpenDatasetStore built-in tool for outlier removal
df = store.modify_outliers(df, columns=cols_to_clean, find_and_delete=True)


🚨 room_1_t: Found 308 outliers.
🚨 room_2_t: Found 198 outliers.
🚨 room_3_t: Found 206 outliers.
🚨 supply_t: Found 80 outliers.
🚨 room_1_h: Found 297 outliers.
🚨 room_2_h: Found 106 outliers.
🚨 outside_c: Found 86 outliers.
🚨 room_2_c: Found 104 outliers.
🚨 room_3_c: Found 353 outliers.
🚨 supply_c: Found 140 outliers.
🗑️ Deleted 949 outlier rows total.


In [3]:
# ==========================================
# Controller Performance Plots
# ==========================================
fig = make_subplots(
    rows=7, cols=1,
    subplot_titles=[
        "Temperature Control (Actual vs Setpoint)",
        "Humidity Control",
        "CO2 Monitoring",
        "Occupancy",
        "AHU Commands (Fan & Mixer)",
        "AHU Commands (Cooler)",
        "AHU Commands (Heater)"
    ],
    vertical_spacing=0.06,
    shared_xaxes=True,
    specs=[
        [{"secondary_y": False}],
        [{"secondary_y": False}],
        [{"secondary_y": False}],
        [{"secondary_y": False}],
        [{"secondary_y": False}],
        [{"secondary_y": False}],
        [{"secondary_y": False}]
    ]
)

# 1. Temperature Control
fig.add_trace(go.Scatter(x=df['timestamp'], y=df['room_1_t'], name="Room 1 Temp", line=dict(color="#3498db")), row=1, col=1)
fig.add_trace(go.Scatter(x=df['timestamp'], y=df['room_2_t'], name="Room 2 Temp", line=dict(color="#9b59b6")), row=1, col=1)
fig.add_trace(go.Scatter(x=df['timestamp'], y=df['room_3_t'], name="Room 3 Temp", line=dict(color="#e67e22")), row=1, col=1)
fig.add_trace(go.Scatter(x=df['timestamp'], y=df['outside_t'], name="Outdoor Temp", line=dict(color="#1abc9c", dash="dot")), row=1, col=1)
fig.add_trace(go.Scatter(x=df['timestamp'], y=df['supply_t'], name="Supply Temp", line=dict(color="#f1c40f", dash="dot")), row=1, col=1)
fig.add_trace(go.Scatter(x=df['timestamp'], y=df['zone_ideal_temp'], name="Ideal Temp Setpoint", line=dict(color="#e74c3c", dash="dash")), row=1, col=1)

# 2. Humidity
fig.add_trace(go.Scatter(x=df['timestamp'], y=df['room_1_h'], name="Room 1 Hum (%)", line=dict(color="#3498db")), row=2, col=1)
fig.add_trace(go.Scatter(x=df['timestamp'], y=df['room_2_h'], name="Room 2 Hum (%)", line=dict(color="#9b59b6")), row=2, col=1)
fig.add_trace(go.Scatter(x=df['timestamp'], y=df['room_3_h'], name="Room 3 Hum (%)", line=dict(color="#e67e22")), row=2, col=1)
fig.add_trace(go.Scatter(x=df['timestamp'], y=df['outside_h'], name="Outdoor Hum (%)", line=dict(color="#1abc9c", dash="dot")), row=2, col=1)
fig.add_trace(go.Scatter(x=df['timestamp'], y=df['supply_h'], name="Supply Hum (%)", line=dict(color="#f1c40f", dash="dot")), row=2, col=1)

ideal_hum = df['zone_ideal_hum'] * 100 if df['zone_ideal_hum'].max() <= 1.0 else df['zone_ideal_hum']
fig.add_trace(go.Scatter(x=df['timestamp'], y=ideal_hum, name="Ideal Hum Setpoint (%)", line=dict(color="#e74c3c", dash="dash")), row=2, col=1)

# 3. CO2
fig.add_trace(go.Scatter(x=df['timestamp'], y=df['room_1_c'], name="Room 1 CO2 (ppm)", line=dict(color="#3498db")), row=3, col=1)
fig.add_trace(go.Scatter(x=df['timestamp'], y=df['room_2_c'], name="Room 2 CO2 (ppm)", line=dict(color="#9b59b6")), row=3, col=1)
fig.add_trace(go.Scatter(x=df['timestamp'], y=df['room_3_c'], name="Room 3 CO2 (ppm)", line=dict(color="#e67e22")), row=3, col=1)
fig.add_trace(go.Scatter(x=df['timestamp'], y=df['outside_c'], name="Outdoor CO2 (ppm)", line=dict(color="#1abc9c", dash="dot")), row=3, col=1)
fig.add_trace(go.Scatter(x=df['timestamp'], y=df['supply_c'], name="Supply CO2 (ppm)", line=dict(color="#f1c40f", dash="dot")), row=3, col=1)
fig.add_trace(go.Scatter(x=df['timestamp'], y=df['zone_ideal_co2'], name="Ideal CO2 Setpoint", line=dict(color="#e74c3c", dash="dash")), row=3, col=1)

# 4. Occupancy
if 'actual_occupancy' in df.columns:
    fig.add_trace(go.Scatter(x=df['timestamp'], y=df['actual_occupancy'], name="Actual Occupancy", line=dict(color="#9b59b6", width=3, shape='hv')), row=4, col=1)

# 5. AHU Commands (Fan & Mixer)
fig.add_trace(go.Scatter(x=df['timestamp'], y=df['fan_cmnd'], name="Controller Fan Cmd %", line=dict(color="#e74c3c", width=2)), row=5, col=1)
fig.add_trace(go.Scatter(x=df['timestamp'], y=df['fan'], name="Hardware Fan Actual %", line=dict(color="rgba(231, 76, 60, 0.4)", width=3)), row=5, col=1)
fig.add_trace(go.Scatter(x=df['timestamp'], y=df['mixer_ratio'], name="Controller Mixer Cmd %", line=dict(color="#3498db", width=2)), row=5, col=1)
fig.add_trace(go.Scatter(x=df['timestamp'], y=df['mixer'], name="Hardware Mixer Actual %", line=dict(color="rgba(52, 152, 219, 0.4)", width=3)), row=5, col=1)

# 6. AHU Commands (Cooler)
fig.add_trace(go.Scatter(x=df['timestamp'], y=df['cooler_cmd'], name="Controller Cooler Cmd", line=dict(color="#3498db", width=2)), row=6, col=1)
if 'coolerState' in df.columns:
    fig.add_trace(go.Scatter(x=df['timestamp'], y=df['coolerState'], name="Hardware Cooler State", line=dict(color="rgba(52, 152, 219, 0.4)", width=3)), row=6, col=1)

# 7. AHU Commands (Heater)
fig.add_trace(go.Scatter(x=df['timestamp'], y=df['heater_cmd'], name="Controller Heater Cmd", line=dict(color="#e74c3c", width=2)), row=7, col=1)
if 'heaterState' in df.columns:
    fig.add_trace(go.Scatter(x=df['timestamp'], y=df['heaterState'], name="Hardware Heater State", line=dict(color="rgba(231, 76, 60, 0.4)", width=3)), row=7, col=1)


# --- Layout Styling ---
fig.update_layout(
    height=1800,
    template="plotly_dark",
    title_text="Controller Performance Monitoring",
    margin=dict(l=30, r=30, t=80, b=30),
    hovermode="x unified"
)

fig.update_yaxes(title_text="Temp (°C)", row=1, col=1)
fig.update_yaxes(title_text="Humidity (%)", row=2, col=1)
fig.update_yaxes(title_text="CO2 (ppm)", row=3, col=1)
fig.update_yaxes(title_text="People", row=4, col=1)
fig.update_yaxes(title_text="Percentage (%)", range=[0, 100], row=5, col=1)
fig.update_yaxes(title_text="Command", row=6, col=1)
fig.update_yaxes(title_text="Command", row=7, col=1)

fig.show()
